In [17]:
import requests
from bs4 import BeautifulSoup
from typing import List, Dict
import re


# -----------------------------
# 1. Fetch HTML
# -----------------------------
def fetch_page(url: str) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; TripReportBot/1.0)"
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.text


# -----------------------------
# 2. Parse posts from page
# -----------------------------
def parse_HST_posts(html: str) -> List[Dict]:
    soup = BeautifulSoup(html, "html.parser")

    posts = []

    # High Sierra Topix uses "postbody" class for content
    post_bodies = soup.find_all("div", class_="postbody")

    for post in post_bodies:
        content_div = post.find("div", class_="content")

        if not content_div:
            continue

        text = content_div.get_text(separator="\n", strip=True)

        posts.append({
            "text": text
        })

    return posts

def parse_BackcountryPost_posts(html: str) -> List[Dict]:
    soup = BeautifulSoup(html, "html.parser")

    posts = []

    # High Sierra Topix uses "postbody" class for content
    post_bodies = soup.find_all("article")
    print(len(post_bodies))
    for post in post_bodies:
        content_div = post.find("div", class_="bbWrapper")

        if not content_div:
            continue

        text = content_div.get_text(separator="\n", strip=True)

        posts.append({
            "text": text
        })
    print(len(posts))
    return posts


# -----------------------------
# 3. Filter for Wind River mentions
# -----------------------------
def filter_wind_river(posts: List[Dict]) -> List[Dict]:
    pattern = re.compile(r"\bwind rivers?\b", re.IGNORECASE)

    filtered = []
    for post in posts:
        if pattern.search(post["text"]):
            filtered.append(post)

    return filtered


# -----------------------------
# 4. Main pipeline
# -----------------------------
def scrape_trip_report(url: str) -> List[Dict]:
    html = fetch_page(url)
    # posts = parse_posts(html)
    posts = parse_BackcountryPost_posts(html)
    # wind_posts = filter_wind_river(posts)

    return posts

In [18]:
# url = "https://www.highsierratopix.com/community/viewtopic.php?t=21021"
url = "https://backcountrypost.com/threads/wind-rivers-2025.11797"

results = scrape_trip_report(url)


8
8


In [19]:
results

[{'text': 'I felt lucky to be out on the trail this year after a major hip injury at the beginning of the year that rendered even a short walk of 20\' impossible. But, PT and a nerve ablation minimized the pain spikes. This year\'s WY trip had wildfires, near and far, and a pesky omega blocking weather pattern - kept spinning off cold rain and snow every few days.  Still, good times were had. The smoke and haze impacted photo quality, but there were still some salvageable shots.\nPART 1 - Ink Wells\nI arrived in Jackson Aug 18 but my nephew Glenn got stuck overnight in Chicago. He\'d been scheduled to arrive an hour before me and the airline played the usual delay game. I went ahead and picked up the car, shopped for supplies and by that time it was apparent that he\'d be delayed overnight. I headed for Dubois to set up my tent and was ready to return if necessary.  First I had to deal with the usual summer Jackson traffic\nBy the time I\'d gotten over Togwotee Pass the airline had can

In [20]:
info = []

for i, post in enumerate(results, 1):
    print(f"\n--- Post {i} ---\n")
    print(post["text"][:500])  # preview
    info.append({"source_url": url,
    "text": post["text"],
    "region_hint": "wind rivers"})


--- Post 1 ---

I felt lucky to be out on the trail this year after a major hip injury at the beginning of the year that rendered even a short walk of 20' impossible. But, PT and a nerve ablation minimized the pain spikes. This year's WY trip had wildfires, near and far, and a pesky omega blocking weather pattern - kept spinning off cold rain and snow every few days.  Still, good times were had. The smoke and haze impacted photo quality, but there were still some salvageable shots.
PART 1 - Ink Wells
I arrived 

--- Post 2 ---

I felt lucky to be out on the trail this year after a major hip injury at the beginning of the year that rendered even a short walk of 20' impossible. But, PT and a nerve ablation minimized the pain spikes. This year's WY trip had wildfires, near and far, and a pesky omega blocking weather pattern - kept spinning off cold rain and snow every few days.  Still, good times were had. The smoke and haze impacted photo quality, but there were still some salvageable s

In [21]:
from openai import OpenAI
import json
import os
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)

def parse_trip_report(text: str) -> dict:
    prompt = f"""
Extract structured fish observations from this trip report. There may be multiple fish observations.
Return a single observation for every lake-species-date pair. Only lake_name and species are required.

Return ONLY valid JSON with a list of objects of the following form:
- lake_name
- species
- count (int/null)
- length (int/null)
- date
- notes

Trip report:
{text}
"""

    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt,
    )

    output_text = response.output[0].content[0].text
    return output_text
    # try:
    #     print(output_text)
    #     return json.loads(output_text)
    # except json.JSONDecodeError:
    #     print("Failed to parse JSON")
    #     return None

response = parse_trip_report(info[0]["text"])
stripped = response[response.find("["): response.rfind("]")+1]
observation_list = json.loads(stripped)

In [22]:
stripped = response[response.find("["): response.rfind("]")+1]
observation_list = json.loads(stripped)
valid_list = []
for obs in observation_list:
    if (obs['species'] is not None and obs['species'] != ''):
        valid_list.append(obs)
valid_list

[{'lake_name': 'Ink Wells',
  'species': 'Fish',
  'count': None,
  'length': None,
  'date': '2023-08-20',
  'notes': 'Bob fished the 2 nearest Ink Well lakes and caught endless amounts of fish.'},
 {'lake_name': 'Lower Jean Lake',
  'species': 'Golden',
  'count': 1,
  'length': 18,
  'date': None,
  'notes': 'Had a small golden on after about 3 casts but it jumped and threw the lure. Finally caught one that did not jump, measured 18" on my rod and released unharmed.'},
 {'lake_name': 'Atlantic Canyon',
  'species': 'Brook Trout',
  'count': None,
  'length': None,
  'date': None,
  'notes': "Usually, the brook trout are easy to catch at this lake, but they weren't that afternoon."}]